# Expedia Applied Science – Identify Fake Job Postings

**Goal:** Predict `is-authentic` (1 = authentic, 0 = fake) for each job posting.

**Deliverable:** `submission.csv` with columns `id`, `is-authentic`.

**Metric:** Accuracy.

This notebook walks through every step with comments, visualizations, and top-20 feature importance.


## Step 0 – Install & imports


In [ ]:
# %pip install pandas numpy scikit-learn matplotlib seaborn -q

from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

ROOT = Path(".").resolve()
DATA = ROOT / "data"
FIGS = ROOT / "figures"
ART = ROOT / "artifacts"
FIGS.mkdir(parents=True, exist_ok=True)
ART.mkdir(parents=True, exist_ok=True)

TARGET = "is-authentic"
ID_COL = "id"
TEXT_COL = "title"
BINARY_COLS = ["wfh", "company logo", "screening round"]
CAT_COLS = [
    "department", "industry", "employment type",
    "required experience", "required education", "country", "has_salary",
]
NUM_COLS = ["title_len", "title_words", "missing_count"]
print("Ready. DATA =", DATA)


## Step 1 – Load train / test / sample_output

Expedia note: **test dtypes may differ from train** (e.g. `wfh` as string `"0"`/`"1"` vs int). We coerce later.


In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_output.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Sample output columns (must match submission):", list(sample.columns))
display(train.head(3))
display(test.head(3))
print("\nTrain dtypes:\n", train.dtypes)
print("\nTest dtypes:\n", test.dtypes)
print("\nLabel balance:\n", train[TARGET].value_counts(normalize=True).round(4))


## Step 2 – Helpers: coerce binaries & extract country

**Why:** Mixed types break models. **Country** is a compact geo signal from `location` (`US, NY, ...` → `US`).


In [ ]:
def coerce_binary(series: pd.Series) -> pd.Series:
    """Safe 0/1 coercion for int/float/str/bool (Expedia train/test dtype mismatch)."""
    if series is None:
        return pd.Series(dtype=int)
    s = series.copy()
    if pd.api.types.is_bool_dtype(s):
        return s.astype(int)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float).ne(0).astype(int)

    cleaned = (
        s.astype(str)
        .str.strip()
        .str.lower()
        .replace({"nan": "0", "none": "0", "null": "0", "": "0"})
    )
    mapping = {
        "1": 1, "1.0": 1, "true": 1, "yes": 1, "y": 1, "t": 1,
        "0": 0, "0.0": 0, "false": 0, "no": 0, "n": 0, "f": 0,
    }
    mapped = cleaned.map(mapping)
    need = mapped.isna()
    if need.any():
        parsed = pd.to_numeric(cleaned[need], errors="coerce").fillna(0)
        mapped = mapped.copy()
        mapped.loc[need] = (parsed != 0).astype(int).values
    return mapped.fillna(0).astype(int)


def extract_country(location: pd.Series) -> pd.Series:
    loc = location.fillna("").astype(str)
    return loc.str.split(",").str[0].str.strip().replace({"": "Unknown"})


print("Helpers defined.")


## Step 3 – Feature engineering

### How features affect the authenticity decision
| Feature | Effect on decision |
|---------|--------------------|
| `title` (TF-IDF) | Scam wording vs normal job titles |
| `company logo` | **Strongest:** missing logo → higher fake risk |
| `screening round` | Questions present → more authentic |
| `wfh` | Remote slightly riskier in this corpus |
| `has_salary` | Salary listed vs blank shifts odds |
| `industry` / `department` / education / experience | Category levels shift class odds |
| `country` | Geography proxy from location |
| `missing_count` | Emptier forms → more fake |
| `title_len` / `title_words` | Very short or stuffed titles stand out |


In [ ]:
def engineer(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["title"] = out["title"].fillna("").astype(str)
    out["location"] = out.get("location", pd.Series([""] * len(out))).fillna("").astype(str)

    for c in ["department", "industry", "employment type", "required experience", "required education"]:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = out[c].astype(str).replace({"nan": np.nan, "None": np.nan})

    for c in BINARY_COLS:
        if c not in out.columns:
            out[c] = 0
        out[c] = coerce_binary(out[c])

    salary = out.get("salary range", pd.Series([np.nan] * len(out)))
    out["has_salary"] = (
        ~salary.isna()
        & (salary.astype(str).str.strip() != "")
        & (salary.astype(str).str.lower() != "nan")
    ).astype(int).astype(str)

    out["country"] = extract_country(out["location"])
    out["title_len"] = out["title"].str.len()
    out["title_words"] = out["title"].str.split().str.len().fillna(0)

    miss_cols = [
        "department", "industry", "employment type",
        "required experience", "required education", "salary range",
    ]
    present = [c for c in miss_cols if c in out.columns]
    out["missing_count"] = (
        out[present].isna().sum(axis=1)
        + out[present].astype(str).isin(["", "nan", "None"]).sum(axis=1)
    )

    for c in CAT_COLS:
        out[c] = out[c].fillna("Missing").astype(str)
    return out


# IMPORTANT: call engineer on each frame separately
train_fe = engineer(train)
test_fe = engineer(test)
feature_cols = [TEXT_COL] + BINARY_COLS + CAT_COLS + NUM_COLS
X = train_fe[feature_cols]
y = train_fe[TARGET].astype(int)
X_test = test_fe[feature_cols]
ids = test_fe[ID_COL].astype(str)

print("X:", X.shape, "| y:", y.shape, "| X_test:", X_test.shape)
display(train_fe[feature_cols + [TARGET]].head(3))


## Step 4 – EDA visualizations


In [ ]:
# 4a – Class balance (majority ~95% authentic → naive baseline is high)
fig, ax = plt.subplots(figsize=(6, 4))
counts = train_fe[TARGET].value_counts().sort_index()
ax.bar(["Fake (0)", "Authentic (1)"], counts.values, color=["#c44e52", "#4c72b0"])
ax.set_title("Label balance: is-authentic")
ax.set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax.text(i, v + max(counts.values) * 0.01, str(v), ha="center")
fig.tight_layout()
fig.savefig(FIGS / "01_class_balance.png", dpi=140)
plt.show()


In [ ]:
# 4b – How binary flags shift P(authentic)
rows = []
for f in BINARY_COLS:
    for val in [0, 1]:
        sub = train_fe[train_fe[f] == val]
        rows.append({"feature": f, "value": str(val), "authentic_rate": sub[TARGET].mean(), "n": len(sub)})
rate_df = pd.DataFrame(rows)
display(rate_df)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=rate_df, x="feature", y="authentic_rate", hue="value", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Authentic rate by binary metadata flags")
ax.set_ylabel("P(is-authentic=1)")
fig.tight_layout()
fig.savefig(FIGS / "02_authentic_rate_by_flags.png", dpi=140)
plt.show()


In [ ]:
# 4c – Missingness vs authenticity
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=train_fe, x=TARGET, y="missing_count", ax=ax)
ax.set_xticklabels(["Fake (0)", "Authentic (1)"])
ax.set_title("Missing field count vs authenticity (emptier → faker)")
fig.tight_layout()
fig.savefig(FIGS / "03_missing_vs_authentic.png", dpi=140)
plt.show()


In [ ]:
# 4d – Industries with highest fake rate
tmp = train_fe.copy()
tmp["industry_f"] = tmp["industry"].fillna("Missing")
ind = (
    tmp.groupby("industry_f")
    .agg(n=(ID_COL, "count"), fake_rate=(TARGET, lambda s: 1 - s.mean()))
    .query("n >= 50")
    .sort_values("fake_rate", ascending=False)
    .head(12)
)
display(ind)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=ind["fake_rate"], y=ind.index, ax=ax, color="#c44e52")
ax.set_title("Industries with highest fake rate (n≥50)")
ax.set_xlabel("P(fake)")
fig.tight_layout()
fig.savefig(FIGS / "04_industry_fake_rate.png", dpi=140)
plt.show()


## Step 5 – Build preprocessing + ExtraTrees model

- **Title** → TF-IDF (1–2 grams)
- **Categoricals** → OneHot (rare levels collapsed)
- **Numerics / binaries** → passed through
- **Classifier:** ExtraTrees (fast, strong accuracy, gives feature importances)


In [ ]:
def build_preprocessor() -> ColumnTransformer:
    text = TfidfVectorizer(max_features=2500, ngram_range=(1, 2), min_df=2, stop_words="english")
    cats = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10, sparse_output=True)),
    ])
    nums = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    return ColumnTransformer(
        transformers=[
            ("title", text, TEXT_COL),
            ("cat", cats, CAT_COLS),
            ("num", nums, NUM_COLS),
            ("bin", "passthrough", BINARY_COLS),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def build_model() -> Pipeline:
    return Pipeline([
        ("pre", build_preprocessor()),
        ("clf", ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=1,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1,
        )),
    ])


model = build_model()
print(model)


## Step 6 – Evaluate (CV + holdout accuracy)


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_tr, y_tr)
va_pred = model.predict(X_va)
print("Holdout accuracy:", round(accuracy_score(y_va, va_pred), 4))
print(classification_report(y_va, va_pred, digits=3))

cm = confusion_matrix(y_va, va_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=ax,
    xticklabels=["Fake", "Authentic"], yticklabels=["Fake", "Authentic"],
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Holdout confusion matrix")
fig.tight_layout()
fig.savefig(FIGS / "05_confusion_matrix.png", dpi=140)
plt.show()


## Step 7 – Top 20 feature importance & decision effects

Higher importance = more influence on `is-authentic` splits inside the trees.


In [ ]:
pre = model.named_steps["pre"]
clf = model.named_steps["clf"]
names = pre.get_feature_names_out()
importances = clf.feature_importances_
order = np.argsort(importances)[::-1][:20]
top20 = pd.DataFrame({"feature": names[order], "importance": importances[order]})

def explain(f: str) -> str:
    if "company logo" in f:
        return "Logo missing raises fake odds; logo present supports authentic."
    if "screening round" in f:
        return "Screening questions present → more authentic; absent → riskier."
    if f.endswith("wfh") or "bin__wfh" in f:
        return "WFH=1 slightly associated with higher fraud rate in this corpus."
    if "missing_count" in f:
        return "More empty fields → more likely fake."
    if "has_salary" in f:
        return "Salary listed vs blank shifts authenticity probability."
    if f.startswith("title__"):
        return f"Title token '{f.split('__', 1)[-1]}' helps separate scam vs real wording."
    if f.startswith("cat__"):
        return f"Category level '{f.split('__', 1)[-1]}' shifts class odds."
    if "title_len" in f or "title_words" in f:
        return "Title length/word count helps spot odd postings."
    return "Numeric/text signal used by trees to split authentic vs fake."

top20["how_it_affects_decision"] = top20["feature"].map(explain)
display(top20)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=top20, x="importance", y="feature", ax=ax, color="#4c72b0")
ax.set_title("Top 20 features by ExtraTrees importance\n(higher → more influence on is-authentic)")
ax.set_xlabel("Mean impurity decrease")
fig.tight_layout()
fig.savefig(FIGS / "06_top20_feature_importance.png", dpi=150)
plt.show()

top20.to_csv(ART / "top20_feature_importance.csv", index=False)
top20.to_csv(ART / "top20_feature_effects.csv", index=False)


## Step 8 – Refit on all train data & write `submission.csv`

Final file must have exactly: **`id`, `is-authentic`** (same as `sample_output.csv`).


In [ ]:
# Fit on full training set
model.fit(X, y)
preds = model.predict(X_test).astype(int)

submission = pd.DataFrame({"id": ids, "is-authentic": preds})
submission = submission[["id", "is-authentic"]]  # enforce column order
submission.to_csv(ROOT / "submission.csv", index=False)

print("Wrote submission.csv", submission.shape)
print(submission.head(10))
print("\nPrediction mix:\n", submission["is-authentic"].value_counts())

# Optional offline score if hidden labels exist (local demo only)
hidden = DATA / "test_labels_hidden.csv"
if hidden.exists():
    labels = pd.read_csv(hidden)
    labels["id"] = labels["id"].astype(str)
    merged = submission.merge(labels, on="id", suffixes=("_pred", "_true"))
    acc = accuracy_score(merged["is-authentic_true"], merged["is-authentic_pred"])
    print(f"\nOffline test accuracy: {acc:.4f}")
else:
    print("\nNo hidden labels file — upload submission.csv to HackerRank.")


## Done

- Run all cells top → bottom
- Download / use `submission.csv`
- Figures saved under `figures/`; importance table under `artifacts/`
